In [3]:
import pandas as pd

player_stats = pd.read_csv("../data/processed/player_season_stats.csv")
raw = pd.read_csv("../data/raw/pga_tour_raw.csv")

In [4]:
raw_post2017 = raw[raw['season'] > 2016]

In [5]:
one_tourney_null_sg = player_stats[
    (player_stats['tournaments_played'] == 1) & (player_stats['avg_sg_total'].isnull())
][['season', 'player_id']]

In [6]:
raw_matches = raw_post2017.merge(
    one_tourney_null_sg,
    left_on=['season', 'player id'],
    right_on=['season', 'player_id']
)

raw_matches[['season', 'player', 'course', 'tournament name', 'sg_total']].head(20)

,season,player,course,tournament name,sg_total
0,2022,Daniel van Tonder,"Southern Hills Country Club - Tulsa, OK",PGA Championship,NaN
1,2022,Jose Maria Olazabal,"Augusta National Golf Club - Augusta, GA",Masters Tournament,NaN
2,2022,Dominic Bozzelli,"Corales Puntacana GC - Punta Cana, Dominican R...",Corales Puntacana Resort & Club Championship,NaN
3,2022,Brett Stegmaier,"Grand Reserve Country Club - Rio Grande, Puert...",Puerto Rico Open,NaN
4,2022,Charlie Beljan,"Grand Reserve Country Club - Rio Grande, Puert...",Puerto Rico Open,NaN
5,2022,Chris Couch,"Grand Reserve Country Club - Rio Grande, Puert...",Puerto Rico Open,NaN
6,2022,Carl Pettersson,"Grand Reserve Country Club - Rio Grande, Puert...",Puerto Rico Open,NaN
7,2022,Josh Teater,"Grand Reserve Country Club - Rio Grande, Puert...",Puerto Rico Open,NaN
8,2022,Smylie Kaufman,"Grand Reserve Country Club - Rio Grande, Puert...",Puerto Rico Open,NaN
9,2022,Spencer Ralston,"Grand Reserve Country Club - Rio Grande, Puert...",Puerto Rico Open,NaN


In [7]:
augusta_check = raw_post2017[raw_post2017['course'].str.contains('Augusta', na=False)]
augusta_check.groupby('season')['sg_total'].apply(lambda x: x.isnull().mean())

season
2017    1.000000
2018    1.000000
2019    1.000000
2021    1.000000
2022    0.071429
Name: sg_total, dtype: float64

Some players with only one tournament played are showing null sg data, this appears to be linked to the 17 courses identified with no sg data (non PGA Tour events that don't have ShotLink technology, e.g. an open tournamnet). The Masters has strokes gained data starting in 2022 which aligns with information found online about when the Masters started adding sg data.

In [8]:
player_stats[player_stats['player'].str.contains('Rory', case=False, na=False)][
    ['season', 'player']
].drop_duplicates()

,season,player
29,2017,Rory McIlroy
222,2017,Rory Sabbatini
332,2018,Rory McIlroy
479,2018,Rory Sabbatini
665,2019,Rory McIlroy
695,2019,Rory Sabbatini
1062,2020,Rory McIlroy
1229,2020,Rory Sabbatini
1443,2021,Rory McIlroy
1714,2021,Rory Sabbatini


In [9]:
player_stats[(player_stats['season'] == 2022) & (player_stats['player'] == 'Rory McIlroy')]

,season,player_id,player,tournaments_played,avg_sg_putt,avg_sg_arg,avg_sg_app,avg_sg_ott,avg_sg_t2g,avg_sg_total,...,avg_sg_t2g_rank,avg_sg_total_rank,sum_sg_putt_rank,sum_sg_arg_rank,sum_sg_app_rank,sum_sg_ott_rank,sum_sg_t2g_rank,sum_sg_total_rank,sg_total_prev_season,sg_total_delta
2168,2022,3470,Rory McIlroy,10,NaN,NaN,NaN,NaN,NaN,NaN,...,190.0,190.0,302,302,302,302,302,302,1.084118,NaN


In [10]:
rory_2022_raw = raw_post2017[
    (raw_post2017['season'] == 2022) & (raw_post2017['player'] == 'Rory McIlroy')
][['tournament name', 'course', 'sg_total', 'sg_putt', 'sg_arg', 'sg_app', 'sg_ott']]

rory_2022_raw

,tournament name,course,sg_total,sg_putt,sg_arg,sg_app,sg_ott
94,The Memorial Tournament pres. by Nationwide,"Muirfield Village Golf Club - Dublin, OH",NaN,NaN,NaN,NaN,NaN
325,PGA Championship,"Southern Hills Country Club - Tulsa, OK",NaN,NaN,NaN,NaN,NaN
601,Wells Fargo Championship,"TPC Potomac at Avenel Farm - Potomac, MD",NaN,NaN,NaN,NaN,NaN
936,Masters Tournament,"Augusta National Golf Club - Augusta, GA",NaN,NaN,NaN,NaN,NaN
1066,Valero Texas Open,"TPC San Antonio - San Antonio, TX",NaN,NaN,NaN,NaN,NaN
1427,The Players Championship,"TPC Sawgrass - Ponte Vedra Beach, FL",NaN,NaN,NaN,NaN,NaN
1623,Arnold Palmer Invitational Pres. by Mastercard,"Bay Hill - Orlando, FL",NaN,NaN,NaN,NaN,NaN
1885,The Genesis Invitational,"Riviera Country Club - Pacific Palisades, CA",NaN,NaN,NaN,NaN,NaN
2634,Hero World Challenge,"Albany - New Providence, Bahamas",NaN,NaN,NaN,NaN,NaN
3253,The CJ Cup @ Summit,"The Summit Club - Las Vegas, NV",NaN,NaN,NaN,NaN,NaN


In [11]:
rory_2022_raw_full = raw_post2017[
    (raw_post2017['season'] == 2022) & (raw_post2017['player'] == 'Rory McIlroy')
]
rory_2022_raw_full[['tournament name', 'strokes', 'made_cut', 'Finish', 'sg_total']]

,tournament name,strokes,made_cut,Finish,sg_total
94,The Memorial Tournament pres. by Nationwide,286,1,NaN,NaN
325,PGA Championship,278,1,NaN,NaN
601,Wells Fargo Championship,276,1,NaN,NaN
936,Masters Tournament,281,1,NaN,NaN
1066,Valero Texas Open,145,0,NaN,NaN
1427,The Players Championship,285,1,NaN,NaN
1623,Arnold Palmer Invitational Pres. by Mastercard,289,1,NaN,NaN
1885,The Genesis Invitational,274,1,NaN,NaN
2634,Hero World Challenge,282,1,NaN,NaN
3253,The CJ Cup @ Summit,263,1,NaN,NaN


In [12]:
wells_fargo_2022 = raw_post2017[
    (raw_post2017['season'] == 2022) & (raw_post2017['tournament name'] == 'Wells Fargo Championship')
]
wells_fargo_2022['sg_total'].isnull().mean()

np.float64(0.014705882352941176)